#   Generación de contenido con IA generativa


**Programa:** Ingeniería de software y datos

**Materia:** Deep learning avanzado

**Grupo:** PREICA2502B020125

**Profesor:** Laura Alejandra Sanchez

**Estudiantes:** Luisa Carvajal, Valentina Ayala, Hugo Carvajal

**Universidad:** Institución Universitaria Digital de Antioquia – IU DIGITAL

In [ ]:
# Importaciones
import os
from google.colab import files
import gradio as gr
import pandas as pd
import random

In [ ]:
# crea una lista con los nombres de todos los archivos necesarios.
# Solo sube si los archivos NO existen ya
archivos_necesarios = [
    "daily_food_nutrition_dataset.csv",
    "salmon_horno.jpg", "huevos_revueltos.jpg", "avena_frutas.jpg",
    "vaso_agua.jpg", "filete_pescado.jpg", "bowl_legumbres.jpg",
    "ensalada_quinoa.jpg", "pollo_parrilla.jpg","fallback_smoothie.jpg","sopa_verduras.jpg"
]

# Detecta qué archivos faltan realmente, para no pedir subir todo otra vez.
faltan = [f for f in archivos_necesarios if not os.path.exists(f)]

if faltan:
    print(f"Faltan {len(faltan)} archivos. Subiéndolos ahora...")
    uploaded = files.upload()
else:
    print("Todos los archivos ya están aquí. ¡Perfecto!")

Faltan 11 archivos. Subiéndolos ahora...


TypeError: 'NoneType' object is not subscriptable

In [ ]:
# Carga y preparación del dataset
df = pd.read_csv("daily_food_nutrition_dataset.csv", on_bad_lines='skip')
df["Calories (kcal)"] = pd.to_numeric(df["Calories (kcal)"], errors="coerce")
df = df.dropna(subset=["Calories (kcal)"])

#  Filtra, ordena y se elige aleatoriamente entre los 15 mejores platos
def elegir_plato_variado(max_kcal):
    opciones = df[df["Calories (kcal)"] <= max_kcal]
    if opciones.empty:
        opciones = df
    # Tomamos los 15 mejores en proteína y elegimos uno al azar.
    top_15 = opciones.sort_values("Protein (g)", ascending=False).head(15)
    return top_15.sample(1).iloc[0]

# Permite que cada plato mostrado tenga una imagen.
def obtener_foto(plato):
    p = str(plato).lower()
    if any(x in p for x in ["salmon", "salmón", "cod", "pescado"]): return "salmon_horno.jpg"
    if any(x in p for x in ["chicken", "pollo", "turkey", "pavo"]): return "pollo_parrilla.jpg"
    if any(x in p for x in ["quinoa", "tofu", "ensalada"]): return "ensalada_quinoa.jpg"
    if any(x in p for x in ["egg", "huevo", "scrambled", "omelette"]): return "huevos_revueltos.jpg"
    if any(x in p for x in ["oatmeal", "avena", "porridge"]): return "avena_frutas.jpg"
    if any(x in p for x in ["soup", "sopa", "lentejas", "lentil"]): return "sopa_verduras.jpg"
    if any(x in p for x in ["legumbres", "frijoles", "bean", "chickpea"]): return "bowl_legumbres.jpg"
    if any(x in p for x in ["yogurt", "smoothie", "batido", "shake"]): return "fallback_smoothie.jpg"
    return "fallback_smoothie.jpg"  # fallback final

# 20 consejos aleatorio.
consejos = [
    "Bebe un vaso de agua antes de cada comida",
    "Camina 10 minutos después de comer",
    "El sueño de 7-8h es el mejor quemador de grasa",
    "Come despacio: tu cerebro tarda 20 min en decir 'lleno'",
    "Evita los jugos: come la fruta entera",
    "Las legumbres son oro puro",
    "El salmón mejora tu ánimo en 2 semanas",
    "El té verde sin azúcar acelera el metabolismo",
    "El vinagre antes de comer reduce el azúcar en sangre",
    "Frutos secos: máximo un puñado al día",
    "El aceite de oliva virgen es medicina",
    "Plato pequeño = comes 22% menos",
    "Respira 5 veces profundo antes de comer",
    "La canela controla el azúcar",
    "El ejercicio de fuerza quema grasa 48h después",
    "Si tienes hambre → pepino o apio",
    "La avena es el mejor desayuno",
    "Agua con limón en ayunas activa tu metabolismo",
    "¡TÚ PUEDES! Cada comida es una victoria",
    "Hoy es el primer día del resto de tu vida saludable"
]

# Es un sistema dinámico para personalizar el plato.
def generar_recomendacion_dia(nivel_obesidad, nivel_calorias, vasos_agua, comidas_dia, edad, ejercicio, favc):

    max_kcal = 600
    if nivel_obesidad == "Obesidad Tipo III": max_kcal = 350
    elif nivel_obesidad == "Sobrepeso Nivel I": max_kcal = 500

    if nivel_calorias == "Muy alto": max_kcal -= 100
    if edad == "Niño (5-12)": max_kcal += 200
    if edad == "Adulto mayor (60+)": max_kcal -= 80
    if ejercicio == "Mucho (6-7 días)": max_kcal += 150

    # Selecciona el plato adecuado.
    comida = elegir_plato_variado(max_kcal)
    plato = comida["Food_Item"]
    kcal = int(comida["Calories (kcal)"])
    proteina = comida["Protein (g)"]

    consejo = random.choice(consejos) # Elige el consejo aleatorio.

    # construye el txto final.
    texto = f"""
# TU RECOMENDACIÓN DEL DÍA

**Perfil:** {edad} • {nivel_obesidad}
**Actividad:** {ejercicio} • {vasos_agua} vasos agua • {comidas_dia} comidas/día

**PLATO DEL DÍA:**
**{plato}**

{kcal} kcal • {proteina}g proteína

**CONSEJO DEL DÍA:**
{consejo}

HOY GANA TU VERSIÓN MÁS SALUDABLE
"""
    # Retorna texto y foto.
    return texto, obtener_foto(plato)

# Interfaz gráfica con GRADIO
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# NUTRIGEN-IA\n### 1 comida + 1 consejo distinto cada vez")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Tus datos")
            nivel_obesidad = gr.Dropdown(["Peso Normal", "Sobrepeso Nivel I", "Obesidad Tipo III"], label="Nivel de obesidad")
            nivel_calorias = gr.Dropdown(["Bajo", "Normal", "Alto", "Muy alto"], label="Consumo calórico")
            vasos_agua = gr.Slider(1, 15, 8, step=1, label="Vasos de agua al día")
            comidas_dia = gr.Slider(2, 6, 3, step=1, label="Comidas al día")
            edad = gr.Radio(["Niño (5-12)", "Adolescente (13-18)", "Adulto (19-59)", "Adulto mayor (60+)"], label="Edad")
            ejercicio = gr.Dropdown(["Nada", "Poco (1-2 días)", "Regular (3-5 días)", "Mucho (6-7 días)"], label="Ejercicio")
            favc = gr.Radio(["Sí", "No"], label="¿Comes comida calórica seguido?")

            btn = gr.Button("NUEVA RECOMENDACIÓN", variant="primary", size="lg")

        with gr.Column(scale=2):
            texto_out = gr.Textbox(label="Tu recomendación del día", lines=16)
            foto_out = gr.Image(label="Tu plato recomendado", height=500)

    # Botón que genera recomendación.
    btn.click(
        fn=generar_recomendacion_dia,
        inputs=[nivel_obesidad, nivel_calorias, vasos_agua, comidas_dia, edad, ejercicio, favc],
        outputs=[texto_out, foto_out]
    )

# Lanzamiento de la aplicación.
app.launch(share=True)

FileNotFoundError: [Errno 2] No such file or directory: 'daily_food_nutrition_dataset.csv'